In [28]:
import pandas as pd
df = pd.read_csv("shopee_reviews.csv", header=1)

print(df.head(10))
df["comment"]

                                             comment
0  Chất lượng sản phẩm: dày dặn Đúng với mô tả: g...
1  Chất lượng sản phẩm: Chưa dùng Đúng với mô tả:...
2  #JISOO ĐƯỢC NHẮC ĐẾN TRONG BÀI VIẾT CỦA VOGUE ...
3  Đúng với mô tả: đẹp Chất lượng sản phẩm: tốt, ...
4  Chất lượng sản phẩm: tam Đúng với mô tả: đúng ...
5  Đúng với mô tả: ko biết Chất lượng sản phẩm: o...
6  Đúng với mô tả: sản phẩm đúng với mô tả Chất l...
7  Chất lượng sản phẩm: ổn Đúng với mô tả: đúng M...
8  Lần sau shop nên xem kỹ trc khi giao, góc trái...
9  Giao hàng nhanh Đóng gói kỹ Tranh nhiều chi ti...


0      Chất lượng sản phẩm: dày dặn Đúng với mô tả: g...
1      Chất lượng sản phẩm: Chưa dùng Đúng với mô tả:...
2      #JISOO ĐƯỢC NHẮC ĐẾN TRONG BÀI VIẾT CỦA VOGUE ...
3      Đúng với mô tả: đẹp Chất lượng sản phẩm: tốt, ...
4      Chất lượng sản phẩm: tam Đúng với mô tả: đúng ...
                             ...                        
287                                    Shop giao lộn mẫu
288                                              Cũng dc
289                                Shop giao thiếu khung
290                                                   Ok
291                                       Ko có sơn bóng
Name: comment, Length: 292, dtype: str

In [41]:
import pandas as pd
import json
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# Khởi tạo client Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 2. Hàm gọi API để trích xuất ABSA
def extract_absa_with_api(comment):
    prompt = f"""
    Mày là một chuyên gia phân tích dữ liệu. Hãy phân tích bình luận Shopee sau đây và trích xuất các khía cạnh (aspect) cùng cảm xúc (sentiment) tương ứng.
    - Aspect CHỈ ĐƯỢC CHỌN từ danh sách này: ["chat_luong","dung_voi_mo_ta","dong_goi", "giao_hang", "gia_ca"]
    - Sentiment CHỈ ĐƯỢC CHỌN từ: ["positive", "negative", "neutral"]
    
    Bình luận: "{comment}"
    
    Trả về ĐÚNG một đối tượng JSON có thuộc tính "results" là một mảng chứa kết quả. 
    Ví dụ: {{"results": [{{"aspect": "dong_goi", "sentiment": "positive"}}, {{"aspect": "chat_luong", "sentiment": "negative"}}]}}
    """
    
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.3-70b-versatile",
            temperature=0,
            response_format={"type": "json_object"},
        )
        response_text = chat_completion.choices[0].message.content
        return json.loads(response_text).get("results", [])
    except Exception as e:
        print(f"Lỗi ở comment: {comment[:30]}... - Chi tiết: {e}")
        return []

# --- CHẠY THỬ VỚI CÂU CỦA MÀY NHÉ ---
test_comment = "Chất lượng sản phẩm: dày dặn Đúng với mô tả: gần giống mô tả Giao hàng nhanh Shop đóng gói cẩn thận Nhưng màu tô thì khác hình minh họa rất nhiều"
print("Đang gọi API chạy thử...")
print(json.dumps(extract_absa_with_api(test_comment), indent=2, ensure_ascii=False))


Đang gọi API chạy thử...
[
  {
    "aspect": "chat_luong",
    "sentiment": "positive"
  },
  {
    "aspect": "dung_voi_mo_ta",
    "sentiment": "neutral"
  },
  {
    "aspect": "giao_hang",
    "sentiment": "positive"
  },
  {
    "aspect": "dong_goi",
    "sentiment": "positive"
  },
  {
    "aspect": "gia_ca",
    "sentiment": "neutral"
  }
]


In [ ]:
# 3. Quét toàn bộ DataFrame
new_rows = []

# Giả sử bảng df của mày có 100 dòng, có thể chạy vòng lặp (nếu nhiều quá thì cắt nhỏ ra test trước)
for index, row in df.iterrows(): # Tạm thời lấy 10 dòng đầu chạy thử: df.head(10).iterrows()
    comment_text = row['comment']
    
    # Gọi API cho từng dòng
    extracted_data = extract_absa_with_api(comment_text)
    
    # Tách thành từng dòng mới
    for item in extracted_data:
        new_rows.append({
            'comment': comment_text,
            'aspect': item['aspect'],
            'sentiment': item['sentiment']
        })

# 4. Gộp thành DataFrame mới chuẩn format
df_clean = pd.DataFrame(new_rows)
df_clean.head(10)

In [44]:
import pandas as pd
import json
import time
import math
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Giả sử df đã có từ các cell trên
# Loại bỏ các dòng không có comment để tránh lỗi
df = df.dropna(subset=['comment']).copy()
df['temp_id'] = range(len(df))

def extract_batch_with_groq(batch_data, retries=3):
    input_json = json.dumps(batch_data, ensure_ascii=False)
    
    prompt = f"""
    Mày là chuyên gia phân tích dữ liệu. Hãy phân tích danh sách các bình luận sau:
    {input_json}
    
    Yêu cầu:
    - aspect CHỈ ĐƯỢC CHỌN từ: ["chat_luong","dung_voi_mo_ta","dong_goi", "giao_hang", "gia_ca"]
    - sentiment CHỈ ĐƯỢC CHỌN từ: ["positive", "negative", "neutral"]
    
    Trả về ĐÚNG định dạng JSON là một OBJECT chứa danh sách kết quả trong thuộc tính "data".
    Ví dụ:
    {{
      "data": [
        {{
          "temp_id": 0,
          "results": [
            {{"aspect": "dong_goi", "sentiment": "positive"}}
          ]
        }}
      ]
    }}
    """
    
    for attempt in range(retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.3-70b-versatile",
                response_format={"type": "json_object"},
                temperature=0
            )
            
            response_text = chat_completion.choices[0].message.content
            data = json.loads(response_text)
            return data.get("data", [])
        except Exception as e:
            print(f"Lỗi khi gọi API (Lần thử {attempt + 1}/{retries}): {e}")
            if "Rate limit" in str(e) or "429" in str(e):
                time.sleep(10) # Bị Rate limit thì đợi lâu hơn
            else:
                time.sleep(3)
            
    return []

batch_size = 15 # Groq giới hạn Tokens Per Minute khá gắt nên mỗi lần chạy 15 câu là an toàn
total_batches = math.ceil(len(df) / batch_size)
new_rows = []

print(f"Bắt đầu chạy tổng cộng {total_batches} batches...")

for i in range(total_batches):
    batch_df = df.iloc[i*batch_size : (i+1)*batch_size]
    batch_input = batch_df[['temp_id', 'comment']].to_dict(orient='records')
    
    print(f"Đang xử lý Batch {i+1}/{total_batches}...")
    batch_results = extract_batch_with_groq(batch_input)
    
    for item in batch_results:
        temp_id = item.get('temp_id')
        results = item.get('results', [])
        
        original_comment_series = df[df['temp_id'] == temp_id]['comment']
        if not original_comment_series.empty:
            comment_text = original_comment_series.values[0]
            for res in results:
                new_rows.append({
                    'comment': comment_text,
                    'aspect': res.get('aspect'),
                    'sentiment': res.get('sentiment')
                })
                
    time.sleep(3) # Nghỉ giữa các lần gọi để xả rate limit

df_clean = pd.DataFrame(new_rows)
df_clean.to_csv('shopee_reviews_ABSA_FAST.csv', index=False, encoding='utf-8-sig')
print(f"Xong! Đã lưu {len(df_clean)} kết quả vào shopee_reviews_ABSA_FAST.csv")
df_clean.head()


Bắt đầu chạy tổng cộng 20 batches...
Đang xử lý Batch 1/20...
Đang xử lý Batch 2/20...
Đang xử lý Batch 3/20...
Đang xử lý Batch 4/20...
Đang xử lý Batch 5/20...
Lỗi khi gọi API (Lần thử 1/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqy2zrave9kvqfn127rf0xmc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98682, Requested 1544. Please try again in 3m15.263999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Lỗi khi gọi API (Lần thử 2/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqy2zrave9kvqfn127rf0xmc` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98670, Requested 1500. Please try again in 2m26.88s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'ty

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Load lại data chuẩn
df = pd.read_csv('shopee_reviews.csv', header=1)
df = df.dropna(subset=['comment']).copy()
df['temp_id'] = range(len(df))

def extract_batch_with_groq(batch_data, retries=3):
    input_json = json.dumps(batch_data, ensure_ascii=False)
    
    prompt = f"""
    Mày là chuyên gia phân tích dữ liệu. Hãy phân tích danh sách các bình luận sau:
    {input_json}
    
    Yêu cầu TỐI QUAN TRỌNG:
    - Phân tích ĐẦY ĐỦ TẤT CẢ các bình luận được giao. KHÔNG ĐƯỢC BỎ SÓT BẤT KỲ temp_id NÀO!
    - aspect CHỈ ĐƯỢC CHỌN từ: ["chat_luong","dung_voi_mo_ta","dong_goi", "giao_hang", "gia_ca"]
    - sentiment CHỈ ĐƯỢC CHỌN từ: ["positive", "negative", "neutral"]
    
    Trả về ĐÚNG định dạng JSON là một OBJECT chứa danh sách kết quả trong thuộc tính "data".
    Ví dụ:
    {{
      "data": [
        {{
          "temp_id": 0,
          "results": [
            {{"aspect": "dong_goi", "sentiment": "positive"}}
          ]
        }}
      ]
    }}
    """
    
    for attempt in range(retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.3-70b-versatile",
                response_format={"type": "json_object"},
                temperature=0
            )
            
            response_text = chat_completion.choices[0].message.content
            data = json.loads(response_text)
            return data.get("data", [])
        except Exception as e:
            print(f"Lỗi khi gọi API (Lần thử {attempt + 1}/{retries}): {e}")
            time.sleep(10)
            
    return []

new_rows = []
queue = df[['temp_id', 'comment']].to_dict(orient='records')
batch_size = 10 # Giảm xuống 10 để AI không bị quá tải dẫn đến lười/bỏ sót
max_iterations = 200
iteration = 0

print(f"Bắt đầu xử lý {len(queue)} comments với cơ chế CHỐNG BỎ SÓT...")

while len(queue) > 0 and iteration < max_iterations:
    iteration += 1
    batch_input = queue[:batch_size]
    queue = queue[batch_size:] # Lấy ra và xóa khỏi queue
    
    expected_ids = set([item['temp_id'] for item in batch_input])
    print(f"[Batch {iteration}] Đang xử lý {len(batch_input)} comments... Còn lại {len(queue)} trong hàng đợi.")
    
    batch_results = extract_batch_with_groq(batch_input)
    
    processed_ids = set()
    for item in batch_results:
        temp_id = item.get('temp_id')
        if temp_id is not None:
            processed_ids.add(temp_id)
        results = item.get('results', [])
        
        original_comment = df[df['temp_id'] == temp_id]['comment']
        if not original_comment.empty:
            comment_text = original_comment.values[0]
            if len(results) == 0:
                new_rows.append({'comment': comment_text, 'aspect': 'other', 'sentiment': 'neutral'})
            else:
                for res in results:
                    new_rows.append({'comment': comment_text, 'aspect': res.get('aspect'), 'sentiment': res.get('sentiment')})
                    
    # Kiểm tra xem AI có bỏ sót comment nào không
    missed_ids = expected_ids - processed_ids
    if len(missed_ids) > 0:
        print(f"⚠️ CẢNH BÁO: AI làm biếng bỏ sót {len(missed_ids)} comments. Đang tự động nhét lại vào hàng đợi!")
        missed_comments = [item for item in batch_input if item['temp_id'] in missed_ids]
        queue = missed_comments + queue # Nhét lại vào đầu queue để xử lý tiếp
        
    time.sleep(3)

df_clean = pd.DataFrame(new_rows)
df_clean.to_csv('shopee_reviews_ABSA_ROBUST.csv', index=False, encoding='utf-8-sig')
print(f"Hoàn thành xuất sắc! Đã lưu {len(df_clean)} kết quả vào shopee_reviews_ABSA_ROBUST.csv")
df_clean.head()


In [45]:
import pandas as pd
import json
import time
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

df = pd.read_csv('shopee_reviews.csv', header=1)
df = df.dropna(subset=['comment']).copy()
df['temp_id'] = range(len(df))

def extract_batch_with_advanced_prompt(batch_data, retries=3):
    input_json = json.dumps(batch_data, ensure_ascii=False)
    
    prompt = f"""
    Mày là chuyên gia phân tích dữ liệu ngôn ngữ tự nhiên (NLP) cho thương mại điện tử.
    Hãy phân tích danh sách các bình luận Shopee dưới đây để tìm ra Khía cạnh (Aspect) và Cảm xúc (Sentiment).
    
    HƯỚNG DẪN ĐỊNH NGHĨA KHÍA CẠNH (ASPECT):
    Khách hàng thường nói bóng gió, không dùng từ khóa trực tiếp. Hãy suy luận theo nguyên tắc sau:
    1. "chat_luong" (Chất lượng sản phẩm): Tranh đẹp/xấu, màu sắc nét/nhạt/khô, chất giấy, dễ tô/khó tô, thiếu màu, cọ vẽ xịn/dởm...
    2. "dung_voi_mo_ta" (Đúng với mô tả): Hàng giao đúng mẫu/sai mẫu, giống/khác hình shop đăng, màu sắc thực tế khác hình minh họa,...
    3. "dong_goi" (Đóng gói): Hàng bị móp méo, rách, bể, gãy, đóng gói cẩn thận, bọc kỹ, sơ sài. (Ví dụ: "ship về tranh bị móp" -> là do dong_goi hoặc giao_hang).
    4. "giao_hang" (Giao hàng): Giao nhanh/chậm, thái độ shipper.
    5. "gia_ca" (Giá cả): Đáng tiền, rẻ, đắt, phù hợp giá tiền.
    
    VÍ DỤ SUY LUẬN (Few-shot learning):
    - "ship về tranh bị móp" -> aspect: "dong_goi", sentiment: "negative"
    - "màu bị khô không tô được" -> aspect: "chat_luong", sentiment: "negative"
    - "màu tô khác hình minh họa rất nhiều" -> aspect: "dung_voi_mo_ta", sentiment: "negative"
    - "quá rẻ cho một bức tranh" -> aspect: "gia_ca", sentiment: "positive"
    - "thiếu cọ vẽ rồi shop ơi" -> aspect: "chat_luong", sentiment: "negative"
    - "màu tô khác hình minh họa rất nhiều" -> aspect: "dung_voi_mo_ta", sentiment: "negative"
    
    Dữ liệu cần phân tích:
    {input_json}
    
    Yêu cầu TỐI QUAN TRỌNG:
    - Phân tích ĐẦY ĐỦ TẤT CẢ các bình luận được giao. KHÔNG ĐƯỢC BỎ SÓT BẤT KỲ temp_id NÀO!
    - Trả về ĐÚNG định dạng JSON là một OBJECT chứa danh sách kết quả trong thuộc tính "data".
    """
    
    for attempt in range(retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model="llama-3.3-70b-versatile",
                response_format={"type": "json_object"},
                temperature=0.1 # Tăng nhẹ temp để AI suy luận tốt hơn một xíu
            )
            
            response_text = chat_completion.choices[0].message.content
            data = json.loads(response_text)
            return data.get("data", [])
        except Exception as e:
            time.sleep(10)
            
    return []

new_rows = []
queue = df[['temp_id', 'comment']].to_dict(orient='records')
batch_size = 10
max_iterations = 200
iteration = 0

print(f"Bắt đầu xử lý với Advanced Prompt (Suy luận ngữ cảnh)...")

while len(queue) > 0 and iteration < max_iterations:
    iteration += 1
    batch_input = queue[:batch_size]
    queue = queue[batch_size:]
    
    expected_ids = set([item['temp_id'] for item in batch_input])
    print(f"[Batch {iteration}] Đang xử lý {len(batch_input)} comments...")
    
    batch_results = extract_batch_with_advanced_prompt(batch_input)
    
    processed_ids = set()
    for item in batch_results:
        temp_id = item.get('temp_id')
        if temp_id is not None: processed_ids.add(temp_id)
        results = item.get('results', [])
        
        original_comment = df[df['temp_id'] == temp_id]['comment']
        if not original_comment.empty:
            comment_text = original_comment.values[0]
            if len(results) == 0:
                new_rows.append({'comment': comment_text, 'aspect': 'other', 'sentiment': 'neutral'})
            else:
                for res in results:
                    new_rows.append({'comment': comment_text, 'aspect': res.get('aspect'), 'sentiment': res.get('sentiment')})
                    
    missed_ids = expected_ids - processed_ids
    if len(missed_ids) > 0:
        missed_comments = [item for item in batch_input if item['temp_id'] in missed_ids]
        queue = missed_comments + queue
        
    time.sleep(3)

df_clean = pd.DataFrame(new_rows)
df_clean.to_csv('shopee_reviews_ABSA_SMART.csv', index=False, encoding='utf-8-sig')
print(f"Hoàn thành xuất sắc! Đã lưu kết quả vào shopee_reviews_ABSA_SMART.csv")
df_clean.head()


Bắt đầu xử lý với Advanced Prompt (Suy luận ngữ cảnh)...
[Batch 1] Đang xử lý 10 comments...
[Batch 2] Đang xử lý 10 comments...
[Batch 3] Đang xử lý 10 comments...
[Batch 4] Đang xử lý 10 comments...
[Batch 5] Đang xử lý 10 comments...
[Batch 6] Đang xử lý 10 comments...
[Batch 7] Đang xử lý 10 comments...
[Batch 8] Đang xử lý 10 comments...
[Batch 9] Đang xử lý 10 comments...
[Batch 10] Đang xử lý 10 comments...
[Batch 11] Đang xử lý 10 comments...
[Batch 12] Đang xử lý 10 comments...
[Batch 13] Đang xử lý 10 comments...
[Batch 14] Đang xử lý 10 comments...
[Batch 15] Đang xử lý 10 comments...
[Batch 16] Đang xử lý 10 comments...
[Batch 17] Đang xử lý 10 comments...
[Batch 18] Đang xử lý 10 comments...
[Batch 19] Đang xử lý 10 comments...
[Batch 20] Đang xử lý 10 comments...
[Batch 21] Đang xử lý 10 comments...
[Batch 22] Đang xử lý 10 comments...
[Batch 23] Đang xử lý 10 comments...
[Batch 24] Đang xử lý 10 comments...
[Batch 25] Đang xử lý 10 comments...
[Batch 26] Đang xử lý 10 co

KeyboardInterrupt: 

In [46]:
# Biến new_rows vẫn còn nguyên trong bộ nhớ!
df_clean_tam = pd.DataFrame(new_rows)

# Lưu ra một tên file mới để tránh nhầm lẫn
df_clean_tam.to_csv('shopee_reviews_ABSA_KHAN_CAP.csv', index=False, encoding='utf-8-sig')

print(f"Đã lưu khẩn cấp thành công {len(df_clean_tam)} dòng dữ liệu!")


Đã lưu khẩn cấp thành công 76 dòng dữ liệu!
